In [1]:
from tensorflow.keras import layers, models

In [2]:
model = models.Sequential()

model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(224, 224, 3)))
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Conv2D(128, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

model.add(layers.Flatten())

model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dropout(0.5))

model.add(layers.Dense(1, activation='sigmoid'))

c:\Users\krish\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [3]:
model.compile(
    optimizer='adam', 
    loss='binary_crossentropy', 
    metrics=['accuracy']
)

In [4]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [7]:
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3, 
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    'best_model.h5', 
    monitor='val_accuracy', 
    save_best_only=True
)

In [10]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)

valid_datagen = ImageDataGenerator(rescale=1./255)

train_data = train_datagen.flow_from_directory(
    "../data/train",
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

valid_data = valid_datagen.flow_from_directory(
    "../data/valid",
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

Found 2662 images belonging to 2 classes.
Found 442 images belonging to 2 classes.


In [14]:
!pip install scipy
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=10,
    callbacks=[early_stop, checkpoint]
)


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl (41.3 MB)
Epoch 1/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5447 - loss: 0.8574

84/84 ━━━━━━━━━━━━━━━━━━━━ 120s 1s/step - accuracy: 0.5800 - loss: 0.7225 - val_accuracy: 0.6923 - val_loss: 0.5670
Epoch 2/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 989ms/step - accuracy: 0.6750 - loss: 0.5963

84/84 ━━━━━━━━━━━━━━━━━━━━ 87s 1s/step - accuracy: 0.7021 - loss: 0.5680 - val_accuracy: 0.7262 - val_loss: 0.5748
Epoch 3/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 987ms/step - accuracy: 0.7466 - loss: 0.5106

84/84 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - accuracy: 0.7660 - loss: 0.4863 - val_accuracy: 0.7353 - val_loss: 0.5235
Epoch 4/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7672 - loss: 0.4659

84/84 ━━━━━━━━━━━━━━━━━━━━ 88s 1s/step - accuracy: 0.7697 - loss: 0.4587 - val_accuracy: 0.7647 - val_loss: 0.4949
Epoch 5/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 986ms/step - accuracy: 0.7874 - loss: 0.4322

84/84 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - accuracy: 0.7885 - loss: 0.4379 - val_accuracy: 0.7760 - val_loss: 0.5399
Epoch 6/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 88s 1s/step - accuracy: 0.8032 - loss: 0.4187 - val_accuracy: 0.7443 - val_loss: 0.4735
Epoch 7/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 976ms/step - accuracy: 0.8041 - loss: 0.4193

84/84 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - accuracy: 0.8017 - loss: 0.4269 - val_accuracy: 0.8032 - val_loss: 0.4170
Epoch 8/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 975ms/step - accuracy: 0.8248 - loss: 0.3929

84/84 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - accuracy: 0.8216 - loss: 0.3867 - val_accuracy: 0.8258 - val_loss: 0.4047
Epoch 9/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 85s 1s/step - accuracy: 0.8373 - loss: 0.3539 - val_accuracy: 0.8122 - val_loss: 0.4240
Epoch 10/10
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 980ms/step - accuracy: 0.8492 - loss: 0.3554

84/84 ━━━━━━━━━━━━━━━━━━━━ 86s 1s/step - accuracy: 0.8441 - loss: 0.3586 - val_accuracy: 0.8484 - val_loss: 0.3872


In [15]:
import pickle

with open("../models/history.pkl", "wb") as f:
    pickle.dump(history.history, f)